### Feature Engineering Techniques

#### 0. Core idea

Transform raw data into a representation that exposes patterns to a model more directly than the raw values do. The same underlying information, reshaped so the model does not have to work as hard (or cannot work hard enough with a simple model like logreg) to find the pattern itself.

This notebook covers categorical encoding, numerical transforms, interaction features, and text/date features. Cyclical (sin/cos) encoding for time features already has its own dedicated notebook in this folder, `cyclical-features.ipynb`, referenced rather than repeated here.


#### 1. Categorical encoding

Toy: `contact_channel` column with values [dating app, email, dating app, social media DM].

One-hot encoding: one binary column per category.
```
              dating_app  email  social_media_DM
dating app        1         0          0
email              0         1          0
dating app        1         0          0
social media DM    0         0          1
```
Simple, no ordering implied, but dimensionality explodes with high-cardinality categories (this project's `impersonated_entity` field, free text, could have dozens of distinct values).

Label/ordinal encoding: one integer per category (dating app=0, email=1, social media DM=2). Compact, but implies a false ordering, a linear model would treat "social media DM" (2) as literally twice "email" (1), meaningless for a truly unordered category. Fine for tree models, which just split on thresholds and do not assume the numbers are meaningfully ordered.

Target/mean encoding: replace each category with the average target value for that category. Powerful (captures the actual relationship, not just an arbitrary label), but leaks the target into the feature if computed carelessly. The CatBoost notebook in this series works through this leakage problem and its fix (Ordered Target Statistics) in full detail with a worked example, same underlying issue as any naive target-mean encoding here.

Frequency encoding: replace each category with how often it appears in the data. Cheap, no leakage risk, loses information about the relationship to the target though, purely about prevalence.


In [ ]:
import pandas as pd

df = pd.DataFrame({"contact_channel": ["dating app", "email", "dating app", "social media DM"]})

print("one-hot:\n", pd.get_dummies(df["contact_channel"]))

print("\nlabel encoding:\n", df["contact_channel"].astype("category").cat.codes)

print("\nfrequency encoding:\n", df["contact_channel"].map(df["contact_channel"].value_counts()))

#### 2. Numerical transforms

Standardization (z-score): (x - mean) / std. Needed for distance-based (KNN, k-means, SVM) and gradient-based (logreg, neural nets) models, not needed for trees, covered with the reasoning already in the KNN and PCA notebooks.

Log transform for skewed distributions. Worked example, amount_lost has values [500, 2500, 10000, 40000], a heavily right-skewed distribution (a few huge values dominate the range):
```
raw values:        [500, 2500, 10000, 40000]
log(raw values):    [6.21, 7.82, 9.21, 10.60]
```
The raw range spans 39,500 (500 to 40000), the log range spans only 4.4 (6.21 to 10.6), the log-transformed feature is far more evenly spread, which helps linear models in particular (a coefficient can now capture "each unit of log-amount matters equally" rather than being dominated by the handful of largest raw values).

Binning: convert a continuous feature into discrete ranges, e.g. amount_lost into [under $1000, $1000-$10000, over $10000]. Trades precision for robustness to outliers and for letting a linear model capture non-linear thresholds it otherwise could not (a single linear coefficient cannot represent "risk jumps at $10000," a binned indicator can).


In [ ]:
import numpy as np

amounts = np.array([500, 2500, 10000, 40000])
log_amounts = np.log(amounts)

print("raw range:", amounts.max() - amounts.min())
print("log range:", (log_amounts.max() - log_amounts.min()).round(2))
print("log values:", log_amounts.round(2))

bins = pd.cut(amounts, bins=[0, 1000, 10000, np.inf], labels=["under_1k", "1k_to_10k", "over_10k"])
print("\nbinned:", list(bins))

#### 3. Interaction features

A single feature might not carry signal alone, but combined with another one it does. Worked example: `urgency_language` (0/1) alone, and `payment_method_requested == gift card` (0/1) alone, might each be weak signals individually, but their interaction (urgency AND gift card requested together) is a much stronger fraud signal than either alone, that specific combination is a very typical high-pressure scam pattern.
```
urgency=1, gift_card=1  -> interaction feature = 1  (both present, strong signal)
urgency=1, gift_card=0  -> interaction feature = 0
urgency=0, gift_card=1  -> interaction feature = 0
urgency=0, gift_card=0  -> interaction feature = 0
```
Linear models (logreg) cannot discover this combination on their own, a single weighted sum of the two features separately cannot represent "only matters when both are true together," you have to engineer the interaction term explicitly and hand it a new column. Tree-based models (the XGBoost, LightGBM, CatBoost notebooks) can discover interactions like this automatically through nested splits, one reason they often need less manual interaction-feature engineering than linear models do.

#### 4. Domain-specific and LLM-derived features

Feature engineering is not always mechanical transforms of existing columns, it can involve genuinely new information extracted from unstructured data. The fraud project's `FEATURE_SCHEMA` (contact_channel, impersonated_entity, payment_method_requested, urgency_language, victim_sent_money) is exactly this: an LLM reads unstructured narrative text and extracts structured features that did not exist as columns before, a form of feature engineering that only became practical with LLMs, previously this kind of signal required manual labeling or complex NLP pipelines (named entity recognition, rule-based extraction) to get anywhere close.


In [ ]:
df_interact = pd.DataFrame({
    "urgency": [1, 1, 0, 0],
    "gift_card": [1, 0, 1, 0],
})
df_interact["urgency_and_giftcard"] = df_interact["urgency"] * df_interact["gift_card"]
print(df_interact)

## Cyclical (Periodic) Features

In [ ]:
import numpy as np
import pandas as pd

# Create sample data
hours = [0, 6, 12, 18, 23, 24]  # 24 wraps to 0
df = pd.DataFrame({'hour': hours})

# Encode cyclically
df['sin_hour'] = np.sin(df['hour'] * 2 * np.pi / 24)
df['cos_hour'] = np.cos(df['hour'] * 2 * np.pi / 24)

print(df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hours = np.arange(24)
sin_hours = np.sin(hours * 2 * np.pi / 24)
cos_hours = np.cos(hours * 2 * np.pi / 24)

plt.figure(figsize=(6,6))
plt.scatter(sin_hours, cos_hours, c=hours, cmap='hsv')
plt.xlabel('sin(hour)')
plt.ylabel('cos(hour)')
plt.title('Cyclical Encoding of Hours')
plt.axis('equal')
for i, h in enumerate(hours):
    plt.annotate(str(h), (sin_hours[i], cos_hours[i]))
plt.show()